# Retrieval Augmented Translation (RAT): French to Kabyle with Gemini

This notebook implements a RAT pipeline using LangChain.
- **Corpus**: `fr_kab.txt` (French-Kabyle parallel sentences)
- **Embedding Model**: `BAAI/bge-m3`
- **Retriever**: `FAISS` via LangChain
- **LLM**: `Google Gemini 2.5 Flash`

In [ ]:
# 1. Install Dependencies (run this first, then restart the kernel)
# !pip install --upgrade langchain langchain-core langchain-google-genai langchain-community sentence-transformers faiss-cpu numpy google-generativeai

In [ ]:
import os
import numpy as np
import pickle
import google.generativeai as genai
from sentence_transformers import SentenceTransformer
import faiss

# Set your Google API key
# Replace with your API key
os.environ["GOOGLE_API_KEY"] = "YOUR_GOOGLE_API_KEY"
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

# Paths for saving/loading
INDEX_PATH = "./faiss_index_gemini.bin"
PAIRS_PATH = "./parallel_pairs_gemini.pkl"

In [ ]:
# 2. Load Parallel Corpus from fr_kab.txt
corpus_path = "./fr_kab.txt"

french_sentences = []
kabyle_sentences = []
parallel_pairs = []  # Store as list of dicts for metadata

with open(corpus_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        parts = line.split('\t')
        parts = [p.strip() for p in parts if p.strip()]
        if len(parts) >= 2:
            french_sentences.append(parts[0])
            kabyle_sentences.append(parts[1])
            parallel_pairs.append({
                "french": parts[0],
                "kabyle": parts[1]
            })

print(f"Loaded {len(french_sentences)} parallel sentence pairs.")
print(f"\nExample pairs:")
for i in range(min(3, len(french_sentences))):
    print(f"  FR: {french_sentences[i]}")
    print(f"  KAB: {kabyle_sentences[i]}\n")

In [ ]:
# 3. Initialize Embedding Model (BGE-M3)
embed_model_name = "BAAI/bge-m3"

# Use SentenceTransformer directly
embeddings = SentenceTransformer(
    embed_model_name, device='cuda')  # Change to 'cpu' if no GPU

print(f"Embedding model loaded: {embed_model_name}")

In [ ]:
# 4. Build or Load FAISS Vector Store

FORCE_REBUILD = False  # Set to True to rebuild even if saved files exist

if os.path.exists(INDEX_PATH) and os.path.exists(PAIRS_PATH) and not FORCE_REBUILD:
    # Load existing index and pairs
    print("Loading saved FAISS index and parallel pairs...")
    index = faiss.read_index(INDEX_PATH)
    with open(PAIRS_PATH, 'rb') as f:
        saved_data = pickle.load(f)
        parallel_pairs = saved_data['parallel_pairs']
        french_sentences = saved_data['french_sentences']
        kabyle_sentences = saved_data['kabyle_sentences']
    print(f"Loaded FAISS index with {index.ntotal} vectors.")
else:
    # Build new index
    print("Building new FAISS index...")

    # Load corpus if not already loaded
    if 'french_sentences' not in dir() or len(french_sentences) == 0:
        corpus_path = "./fr_kab.txt"
        french_sentences = []
        kabyle_sentences = []
        parallel_pairs = []

        with open(corpus_path, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split('\t')
                parts = [p.strip() for p in parts if p.strip()]
                if len(parts) >= 2:
                    french_sentences.append(parts[0])
                    kabyle_sentences.append(parts[1])
                    parallel_pairs.append({
                        "french": parts[0],
                        "kabyle": parts[1]
                    })

    # Generate embeddings
    print("Generating embeddings for French sentences...")
    french_embeddings = embeddings.encode(
        french_sentences,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )
    print(f"Embeddings shape: {french_embeddings.shape}")

    # Build FAISS index
    d = french_embeddings.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(french_embeddings.astype('float32'))

    # Save index and pairs
    print("Saving FAISS index and parallel pairs...")
    faiss.write_index(index, INDEX_PATH)
    with open(PAIRS_PATH, 'wb') as f:
        pickle.dump({
            'parallel_pairs': parallel_pairs,
            'french_sentences': french_sentences,
            'kabyle_sentences': kabyle_sentences
        }, f)
    print(f"Saved to {INDEX_PATH} and {PAIRS_PATH}")

print(f"FAISS index ready with {index.ntotal} vectors.")


def retrieve_examples(query_fr, k=5):
    """Retrieves top-k similar French sentences and their Kabyle translations."""
    query_embedding = embeddings.encode(
        [query_fr], convert_to_numpy=True, normalize_embeddings=True)
    distances, indices = index.search(query_embedding.astype('float32'), k)

    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(parallel_pairs):
            results.append({
                "french": parallel_pairs[idx]["french"],
                "kabyle": parallel_pairs[idx]["kabyle"],
                "score": float(distances[0][i])
            })
    return results

In [ ]:
# 5. Initialize Google Gemini 2.5 Flash (using google-generativeai directly)
model = genai.GenerativeModel('gemini-2.5-flash-preview-05-20')

print("Gemini 2.5 Flash model initialized.")

In [ ]:
# 6. Create RAT Prompt Template
RAT_PROMPT_TEMPLATE = """You are an expert translator specializing in French to Kabyle (Taqbaylit) translation.

Here are some example translations from French to Kabyle for reference:

{examples}

Now translate the following French sentence to Kabyle. Use the examples above to guide your translation style and vocabulary.

French: {query}

Provide only the Kabyle translation, nothing else."""

print("RAT prompt template created.")

In [ ]:
# 7. Helper Functions

def format_retrieved_examples(docs):
    """Format retrieved documents as translation examples."""
    examples = []
    for i, doc in enumerate(docs, 1):
        examples.append(
            f"{i}. French: {doc['french']}\n   Kabyle: {doc['kabyle']}")
    return "\n\n".join(examples)


print("Helper functions created successfully.")

In [ ]:
# 8. Translation Function with Display

def translate_with_rat(french_text: str, show_examples: bool = True):
    """
    Translate French to Kabyle using Retrieval Augmented Translation.
    """
    print("=" * 70)
    print(f"📝 INPUT (French): {french_text}")
    print("=" * 70)

    # Step 1: Retrieve similar examples
    examples = retrieve_examples(french_text, k=5)
    formatted_examples = format_retrieved_examples(examples)

    if show_examples and examples:
        print(f"\n📚 Retrieved Translation Examples:")
        for i, ex in enumerate(examples, 1):
            print(f"  [{i}] (score: {ex['score']:.3f})")
            print(f"      FR:  {ex['french']}")
            print(f"      KAB: {ex['kabyle']}")
        print()

    # Step 2: Generate translation with Gemini
    prompt = RAT_PROMPT_TEMPLATE.format(
        examples=formatted_examples, query=french_text)
    response = model.generate_content(prompt)
    translation = response.text.strip()

    print(f"🔄 Gemini Translation: {translation}")
    print("=" * 70 + "\n")

    return {
        "input": french_text,
        "translation": translation,
        "retrieved_examples": examples
    }

In [ ]:
# 9. Example Usage

# Example 1: Simple greeting
translate_with_rat("Bonjour, comment vas-tu ?")

# Example 2: Statement about language
translate_with_rat("La langue kabyle est très belle.")

# Example 3: Question
translate_with_rat("Où est ta maison ?")

# Example 4: Complex sentence
translate_with_rat("Je veux apprendre à parler kabyle couramment.")

In [ ]:
# 10. Batch Translation Function

def batch_translate(sentences: list, show_examples: bool = False):
    """Translate multiple sentences."""
    results = []
    for sentence in sentences:
        result = translate_with_rat(sentence, show_examples=show_examples)
        results.append({
            "french": sentence,
            "kabyle": result["translation"]
        })
    return results


# Example batch translation
test_sentences = [
    "Merci beaucoup.",
    "Je t'aime.",
    "Quel temps fait-il aujourd'hui ?"
]

print("\n" + "=" * 70)
print("BATCH TRANSLATION")
print("=" * 70)
batch_results = batch_translate(test_sentences, show_examples=False)

In [ ]:
# 11. Interactive Translation (Optional)

def interactive_translate():
    """Run interactive translation session."""
    print("=" * 70)
    print("RAT French → Kabyle Translator (Gemini 2.5 Flash)")
    print("Type 'quit' to exit")
    print("=" * 70)

    while True:
        user_input = input("\nEnter French text: ").strip()
        if user_input.lower() == 'quit':
            print("Goodbye! / Ar tufat!")
            break
        if user_input:
            translate_with_rat(user_input)

# Uncomment to run interactive mode:
# interactive_translate()

In [ ]:
# 12. Quick Translation (Simple Input/Output)

def quick_translate(french_text: str):
    """Quick translation - just input and output, no verbose display."""
    examples = retrieve_examples(french_text, k=5)
    formatted_examples = format_retrieved_examples(examples)

    prompt = RAT_PROMPT_TEMPLATE.format(
        examples=formatted_examples, query=french_text)
    response = model.generate_content(prompt)
    translation = response.text.strip()

    print(f"FR:  {french_text}")
    print(f"KAB: {translation}")
    return translation


# ========== ENTER YOUR TEXT HERE ==========
text_to_translate = "Bonjour, comment ça va ?"
# ==========================================

quick_translate(text_to_translate)

## How This RAT Pipeline Works

1. **Retrieval**: When you input a French sentence, BGE-M3 embeddings find semantically similar French sentences from the parallel corpus.

2. **Context Building**: The retrieved French-Kabyle pairs are formatted as few-shot examples for the LLM.

3. **Generation**: Gemini 2.5 Flash uses these examples to understand Kabyle translation patterns and generates an appropriate translation.

### Key Advantages over NLLB
- **Few-shot learning**: Gemini can learn from examples in the prompt
- **Better context understanding**: Uses retrieved examples to match style and vocabulary
- **More flexible**: Can handle various sentence structures and idioms
- **Larger context window**: Can include more examples for better accuracy

### Configuration Notes
- Replace `YOUR_GOOGLE_API_KEY` with your actual Google AI API key
- Adjust `temperature` (0.0-1.0) for more/less creative translations
- Modify `k` in retriever for more/fewer examples (default: 5)